# Lie Symmetry Analysis of the Korteweg–de Vries (KdV) Equation

This tutorial demonstrates how to use **`symlie`** for symbolic Lie group analysis on jet spaces using SymPy.

We investigate the **Korteweg–de Vries (KdV)** equation:
$$\Delta = u_t + u u_x + u_{xxx} = 0$$

We will:
1. Define the differential equation on jet space.

2. Compute its Fréchet derivative (linearization) and formal adjoint.

3. Derive the determining equations for point symmetries.

4. Calculate the 4-dimensional Lie symmetry algebra (translations, Galilean boost, scaling).

5. Verify generator invariance and compute the Lie bracket commutator table.

6. Perform symmetry reduction to obtain the famous **1-soliton** traveling wave solution.

In [ ]:
import sympy as sp

from symlie import (
    adjoint_frechet_derivative,
    determining_equations,
    frechet_derivative,
    infinitesimals,
    lie_bracket,
    max_derivative_order,
    verify_generator,
)

# Enable pretty printing for SymPy
sp.init_printing()

# Define independent variables and dependent field
x, t = sp.symbols("x t")
u = sp.Function("u")(x, t)

# Define the KdV equation
kdv = u.diff(t) + u * u.diff(x) + u.diff(x, 3)
sp.Eq(kdv, 0)

## 1. Derivative Order and Linearization (Fréchet Derivative)

The maximum derivative order of the KdV equation on the jet space is 3.

In [ ]:
order = max_derivative_order(kdv, u, (x, t))
print(f"Maximum derivative order of KdV: {order}")

The **Fréchet derivative** (linearization operator) acting on an evolutionary test function $Q(x, t)$ is:
$$D_{\text{KdV}}(Q) = Q_t + u Q_x + u_x Q + Q_{xxx}$$

Its **formal adjoint** $D^*_{\text{KdV}}(v)$ under integration by parts is:
$$D^*_{\text{KdV}}(v) = -v_t - u v_x - v_{xxx}$$

In [ ]:
Q = sp.Function("Q")(x, t)
v = sp.Function("v")(x, t)

D_kdv = frechet_derivative(kdv, u, (x, t), Q)
D_star_kdv = adjoint_frechet_derivative(kdv, u, (x, t), v)

print("Fréchet Derivative D_KdV(Q):")
display(D_kdv)

print("Formal Adjoint D*_KdV(v):")
display(D_star_kdv[0])

## 2. Determining Equations for Point Symmetries

A point transformation generator has the form:
$$X = \xi^x(x, t, u) \frac{\partial}{\partial x} + \xi^t(x, t, u) \frac{\partial}{\partial t} + \phi^u(x, t, u) \frac{\partial}{\partial u}$$

The invariance condition $\text{pr}^{(3)} X(\Delta)\big|_{\Delta = 0} = 0$ yields an overdetermined system of linear PDEs for the unknown coefficient functions $\xi^x, \xi^t, \phi^u$.

In [ ]:
det_system = determining_equations(kdv, u, (x, t))

print(f"Total number of determining equations: {len(det_system.equations)}")
for i, eq in enumerate(det_system.equations, 1):
    display(eq)

## 3. Exact Lie Point Symmetry Generators

Solving the determining equations with `infinitesimals` finds the finite-dimensional Lie point symmetry algebra.

In [ ]:
solution = infinitesimals(kdv, u, (x, t), ansatz_degree=1)
print(f"Dimension of the Lie symmetry algebra: {solution.dimension}\n")

labels = [
    "X_1 (Space Translation):",
    "X_2 (Time Translation):",
    "X_3 (Galilean Boost):",
    "X_4 (Scaling / Dilation):",
]

for label, gen in zip(labels, solution.basis):
    is_valid = verify_generator(kdv, u, (x, t), gen)
    print(f"{label}")
    print(f"  xi^x = {gen.xi[0]},  xi^t = {gen.xi[1]},  phi^u = {gen.phi[0]}")
    print(f"  Verified invariant: {is_valid}\n")

print("General Infinitesimal Generator:")
display(solution.general)

## 4. Lie Algebra Structure & Commutator Table

We can calculate the Lie brackets (commutators) $[X_i, X_j]$ between the basis generators using `lie_bracket`.

In [ ]:
basis = solution.basis
n = len(basis)

print("Commutator Table [X_i, X_j]:")
for i in range(n):
    for j in range(i + 1, n):
        bracket = lie_bracket(basis[i], basis[j], u, (x, t))
        print(f"[X_{i + 1}, X_{j + 1}] = xi: {bracket.xi}, phi: {bracket.phi}")

## 5. Symmetry Reduction: The Solitary Wave (1-Soliton) Solution

Using the traveling wave generator $X = c \frac{\partial}{\partial x} + \frac{\partial}{\partial t}$ (a combination of time and space translations with wave speed $c$):

The similarity variable is $z = x - c t$, and the similarity ansatz is $u(x, t) = f(z)$.

Substituting into KdV yields the ordinary differential equation:
$$-c f' + f f' + f''' = 0$$

Integrating once with vanishing boundary conditions at infinity ($f, f', f'' \to 0$ as $|z| \to \infty$):
$$f'' - c f + \frac{1}{2} f^2 = 0$$

Multiplying by $f'$ and integrating again gives the exact **1-soliton solution**:
$$u(x, t) = 3 c \, \text{sech}^2\left(\frac{\sqrt{c}}{2} (x - c t - x_0)\right)$$

In [ ]:
c, x0 = sp.symbols("c x0", positive=True)

# 1-Soliton exact analytical solution
soliton_sol = 3 * c * sp.sech(sp.sqrt(c) / 2 * (x - c * t - x0)) ** 2

print("Exact KdV 1-Soliton Solution u(x, t):")
display(soliton_sol)

# Verify that the soliton satisfies the KdV equation identically
residual = kdv.subs(u, soliton_sol).doit()
residual_simplified = sp.simplify(residual)

print("Substitution residual in KdV equation:")
display(residual_simplified)
assert residual_simplified == 0
print("Verification: Soliton exactly solves the KdV equation!")